# 01 · Prepare source data with separate roles

**Goal:** create labeled training lessons and independent data that
measure what each lesson actually teaches.

**Question:** can a short training response tell us which synthetic lesson
will improve an unfamiliar pose estimator on real video?

The source sequence is **00 → 01 → 02 → 03 → 07**. Run 02 once per configured
source student. After the source decision, prepare independent GAVD references
in 04, deploy without reading their labels in 05, and exchange selected lessons
in 08. Explicitly evaluate in 06. These notebooks contain no precomputed
research results or substitute models.

[Proposal](../../notes/research-agenda/proposals/synthetic-training-selection.md)
· [Notebook guide](README.md)
· [HAIC setup and launch commands](../../slurm/synthetic-training/README.md)

In [ ]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter

root_override = os.environ.get("GAVD6_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
os.environ.setdefault("GAVD6_ROOT", str(PROJECT_ROOT))
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, SVG, display
from gavd6_sjepa.research_directions.synthetic_training.config import RunConfig
from gavd6_sjepa.research_directions.synthetic_training import workflow

cfg = RunConfig.from_env()
RUN_ROOT = cfg.root
get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

def show_result(result):
    # Display the tables and artifact paths returned by a workflow stage.
    if isinstance(result, pd.DataFrame):
        display(result)
    elif isinstance(result, dict):
        for name, value in result.items():
            display(Markdown(f"### {name.replace('_', ' ')}"))
            if isinstance(value, pd.DataFrame):
                display(value)
            elif isinstance(value, Path) and value.suffix == ".svg" and value.is_file():
                display(SVG(filename=str(value)))
            else:
                print(json.dumps(value, indent=2, default=str) if isinstance(value, (list, dict)) else value)
    else:
        print(result)

print(f"Run: {RUN_ROOT}")
print(f"Context representation: {cfg.context_kind}; device: {cfg.device}")

## 1. Separate training from measurement

The common probe, eight lessons, and labeled diagnostic bank use
different source-training motions. Context clips tell the selector
which environment a student will face. Reference clips measure the
resulting accuracy. The context and reference motions are separate;
changing a camera or texture does not create a new independent motion.

The existing AMASS subject registry and split assignments determine
eligibility. Validation contexts and references use excluded people.
Every rendering of a motion keeps its original role.

In [ ]:
display(pd.DataFrame([
    ("probe", "One shared short update", "Labels train the student"),
    ("lesson", "Candidate training choices", "Labels train the student"),
    ("diagnostic", "Known synthetic weaknesses", "Error summaries are selector inputs"),
    ("context", "Simulated deployment collection", "Reference coordinates are not selector inputs"),
    ("reference", "Measure lesson utility", "Errors supply source teacher targets only"),
    ("COCO replay", "Retain real-image pose competence", "Labels train the student"),
], columns=["Data role", "Purpose", "Use of labels"]))

## 2. Render the lesson library and prepare replay

Each rendered RGB image has a fixed person crop, projected landmarks,
visibility flags, and its motion/person provenance. Texture, background,
and viewpoint variation should not accidentally identify the answer.
Full-body AMASS meshes support shoulders, arms, hips, knees, and ankles.

The renderer requires compatible UV topology and real texture images;
it does not silently replace these with a colored stick figure. View,
resolution, blur, and occlusion are controlled appearance conditions,
not simulated clinical diagnoses.

COCO replay is a fixed set of labeled real examples. At the default
mixture, synthetic branches use 90% replay and 10% lesson images.
Replay-only fills every slot with real images, so it sees more real
examples at the same total update budget.

In [ ]:
started = perf_counter()
prepared = workflow.prepare_data(cfg)
show_result(prepared)
print(f"Data preparation took {(perf_counter() - started) / 60:.1f} minutes.")

In [ ]:
# Inspect saved assignments before spending GPU time on teaching trials.
for name in ("synthetic.csv", "replay.csv"):
    path = RUN_ROOT / "data" / name
    if path.is_file():
        table = pd.read_csv(path)
        display(Markdown(f"### {name}: {len(table):,} frames"))
        display(table.head(8))
        groups = [c for c in ("role", "domain_id", "lesson_id") if c in table]
        if groups:
            display(table.groupby(groups, dropna=False).size().rename("frames").to_frame())

## 3. Check the visual task

Inspect representative images and labels in the prepared data folder.
Are body landmarks visible at the intended resolution? Do rendered
shoulders and hips follow the same convention as real COCO landmarks?
Do viewpoint and appearance changes preserve meaningful pose labels?

Short pilot budgets measure whether the experiment runs and whether
choices differ. They are not validated training recipes. Fix any
convention or rendering problems before collecting the utility table.

In [ ]:
from gavd6_sjepa.research_directions.synthetic_training.data import PoseFrameDataset, load_pose_manifest

source_index = load_pose_manifest(RUN_ROOT / "data/synthetic.csv")
examples = source_index.loc[source_index.role.eq("lesson")].groupby("lesson_id", sort=True).head(1)
gallery = PoseFrameDataset(examples)
columns = 4
rows = (len(gallery) + columns - 1) // columns
fig, axes = plt.subplots(rows, columns, figsize=(12, 3.5 * rows), squeeze=False)
for position, axis in enumerate(axes.flat):
    axis.axis("off")
    if position >= len(gallery):
        continue
    sample = gallery[position]
    visible = sample["visible"]
    axis.imshow(sample["image"])
    axis.scatter(*sample["keypoints"][visible].T, s=15, c="#f43f5e", edgecolors="white", linewidths=.4)
    axis.set_title(str(sample["metadata"]["lesson_id"]))
fig.tight_layout()
plt.show()

Next: [02 · Measure source trials](02_measure_source_trials.ipynb),
once per train/validation student.